# Notebook 24 — Effective Noise Direction and Sensitivity

This notebook upgrades the effective-noise-coordinate picture by asking two sharper questions:

1. **Is the fitted effective direction stable across multiple order parameters?**
2. **Where along the effective coordinate does phase-lock quality degrade fastest?**

Instead of only saying that the noisy phase-locked CZ regime can be approximately reduced to

`gamma_eff = gamma + lambda * gamma_phi`

this notebook:
- fits `lambda` for multiple weighted order parameters,
- compares those fitted directions,
- builds 1D response curves,
- computes first and second derivatives,
- identifies the rapid-degradation band along the effective coordinate.

This is the natural next step after Notebooks 21–23 because it turns a visual reduction into a directional and sensitivity analysis.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib', 'scipy']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar
from qutip import basis, qeye, tensor, sigmax, mesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Phase-locked protocol

In [ ]:
opt = {
    'T': 26.0,
    'alpha': 0.10,
    'omega_max': 1.09,
    'delta0': 1.00,
    'p': 1.00,
    'q': 0.72,
}
V = 4.0

for k, v in opt.items():
    print(f"{k}: {v:.6f}")


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)
    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)
    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Collapse operators

In [ ]:
def collapse_ops(gamma_decay=0.0, gamma_phi=0.0):
    ops = []
    if gamma_decay > 0:
        ops.append(np.sqrt(gamma_decay) * sm1)
        ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        ops.append(np.sqrt(gamma_phi) * n1)
        ops.append(np.sqrt(gamma_phi) * n2)
    return ops


## Noisy evolution and effective map

In [ ]:
def run_noisy_evolution(psi0, T, omega_max, alpha, V, delta0, p, q,
                        gamma_decay=0.0, gamma_phi=0.0, n_steps=220):
    H = build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q)
    times = np.linspace(0.0, T, n_steps)
    c_ops = collapse_ops(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    result = mesolve(H, psi0, times, c_ops)
    return result.states[-1]

def state_to_column_mixed(state):
    vals = []
    for basis_state in basis_states:
        if state.isket:
            amp = basis_state.overlap(state)
            vals.append(amp)
        else:
            val = (basis_state.dag() * state * basis_state)
            vals.append(np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val))
    return np.array(vals, dtype=complex)

def build_noisy_effective_map(T, omega_max, alpha, V, delta0, p, q,
                              gamma_decay=0.0, gamma_phi=0.0, n_steps=220):
    cols = []
    finals = []
    for psi0 in basis_states:
        final_state = run_noisy_evolution(
            psi0, T, omega_max, alpha, V, delta0, p, q,
            gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=n_steps
        )
        cols.append(state_to_column_mixed(final_state))
        finals.append(final_state)
    return np.column_stack(cols), finals


## Diagnostics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j*b), np.exp(1j*a), np.exp(1j*(a+b))]).astype(complex)

def compensate_local_z(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    U2 = U1 @ local_z_diagonal(-a, -b)
    return U2, global_phase, a, b, phi_ent

def compensated_cz_fidelity(U_eff):
    U_comp, _, _, _, _ = compensate_local_z(U_eff)
    return process_fidelity(U_comp, U_cz)

def leakage_from_finals(final_states):
    scores = []
    for idx, final_state in enumerate(final_states):
        basis_state = basis_states[idx]
        if final_state.isket:
            amp = basis_state.overlap(final_state)
            score = np.abs(amp) ** 2
        else:
            val = (basis_state.dag() * final_state * basis_state)
            score = np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val)
        scores.append(score)
    return float(1.0 - np.mean(scores))

def coherence_proxy(final_states):
    vals = []
    for state in final_states:
        rho = state.proj() if state.isket else state
        arr = rho.full()
        off = arr.copy()
        np.fill_diagonal(off, 0.0)
        vals.append(np.mean(np.abs(off)))
    return float(np.mean(vals))


## Noise scan

In [ ]:
gamma_decay_vals = np.linspace(0.0, 0.12, 25)
gamma_phi_vals = np.linspace(0.0, 0.12, 25)

cz_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
coh_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
leak_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))

for i, gamma_phi in enumerate(gamma_phi_vals):
    for j, gamma_decay in enumerate(gamma_decay_vals):
        U_eff, finals = build_noisy_effective_map(
            opt['T'], opt['omega_max'], opt['alpha'], V, opt['delta0'], opt['p'], opt['q'],
            gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=200
        )
        cz_map[i, j] = compensated_cz_fidelity(U_eff)
        coh_map[i, j] = coherence_proxy(finals)
        leak_map[i, j] = leakage_from_finals(finals)


## Weighted order parameters

In [ ]:
S_maps = {
    'Balanced': cz_map * coh_map * (1.0 - leak_map),
    'Fidelity-heavy': (cz_map**2) * coh_map * (1.0 - leak_map),
    'Leakage-heavy': cz_map * coh_map * ((1.0 - leak_map)**2),
    'Strict': (cz_map**2) * coh_map * ((1.0 - leak_map)**2),
}

S_norm_maps = {}
for name, arr in S_maps.items():
    S_norm_maps[name] = arr / np.max(arr) if np.max(arr) > 0 else arr.copy()


## Fit effective direction for each metric

In [ ]:
def fit_lambda_for_map(S_norm):
    S_gamma = S_norm[0, :]
    S_phi = S_norm[:, 0]
    interp_gamma = interp1d(gamma_decay_vals, S_gamma, kind='linear', fill_value='extrapolate')

    def loss(lam):
        gamma_eff_phi = lam * gamma_phi_vals
        pred = interp_gamma(np.clip(gamma_eff_phi, gamma_decay_vals.min(), gamma_decay_vals.max()))
        return float(np.mean((pred - S_phi) ** 2))

    fit = minimize_scalar(loss, bounds=(0.1, 5.0), method='bounded')
    return float(fit.x), float(fit.fun), S_gamma, S_phi

fit_results = {}
for name, S_norm in S_norm_maps.items():
    lam, loss, S_gamma, S_phi = fit_lambda_for_map(S_norm)
    fit_results[name] = {
        'lambda': lam,
        'loss': loss,
        'S_gamma': S_gamma,
        'S_phi': S_phi,
    }
    print(name, "lambda =", lam, "axis loss =", loss)


## Compare fitted effective directions

In [ ]:
names = list(fit_results.keys())
lambdas = [fit_results[name]['lambda'] for name in names]
losses = [fit_results[name]['loss'] for name in names]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))

axes[0].bar(names, lambdas)
axes[0].set_ylabel('best-fit lambda')
axes[0].set_title('Effective noise direction by metric')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(names, losses)
axes[1].set_ylabel('axis-slice MSE')
axes[1].set_title('Axis-slice collapse loss by metric')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


## Build 1D response curves along gamma_eff

In [ ]:
response_curves = {}

for name, S_norm in S_norm_maps.items():
    lam = fit_results[name]['lambda']
    gamma_eff_grid = np.zeros_like(S_norm)
    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            gamma_eff_grid[i, j] = gamma_decay + lam * gamma_phi

    gamma_eff_flat = gamma_eff_grid.ravel()
    S_flat = S_norm.ravel()

    order = np.argsort(gamma_eff_flat)
    gamma_eff_sorted = gamma_eff_flat[order]
    S_sorted = S_flat[order]

    n_bins = 24
    bins = np.linspace(gamma_eff_sorted.min(), gamma_eff_sorted.max(), n_bins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    means = np.full(n_bins, np.nan)

    for k in range(n_bins):
        mask = (gamma_eff_sorted >= bins[k]) & (gamma_eff_sorted < bins[k+1])
        if np.any(mask):
            means[k] = np.mean(S_sorted[mask])

    valid = ~np.isnan(means)
    response_curves[name] = {
        'gamma_eff_grid': gamma_eff_grid,
        'gamma_eff_flat': gamma_eff_flat,
        'centers': centers[valid],
        'means': means[valid],
        'interp': interp1d(centers[valid], means[valid], kind='linear', fill_value='extrapolate')
    }


## Compare 1D response curves

In [ ]:
plt.figure(figsize=(7.4, 5.0))
for name in names:
    centers = response_curves[name]['centers']
    means = response_curves[name]['means']
    plt.plot(centers, means, label=name)

plt.xlabel('gamma_eff')
plt.ylabel('Normalized order parameter')
plt.title('Effective-noise response curves')
plt.legend()
plt.tight_layout()
plt.show()


## Sensitivity and curvature

In [ ]:
sensitivity_data = {}

for name in names:
    x = response_curves[name]['centers']
    y = response_curves[name]['means']
    dy = np.gradient(y, x)
    d2y = np.gradient(dy, x)

    idx_peak_drop = int(np.argmin(dy))
    idx_peak_curv = int(np.argmin(d2y))

    sensitivity_data[name] = {
        'x': x,
        'y': y,
        'dy': dy,
        'd2y': d2y,
        'peak_drop_x': float(x[idx_peak_drop]),
        'peak_drop_val': float(dy[idx_peak_drop]),
        'peak_curv_x': float(x[idx_peak_curv]),
        'peak_curv_val': float(d2y[idx_peak_curv]),
    }

    print(name,
          "peak drop at gamma_eff =", sensitivity_data[name]['peak_drop_x'],
          "peak curvature at gamma_eff =", sensitivity_data[name]['peak_curv_x'])


## Plot first derivatives

In [ ]:
plt.figure(figsize=(7.4, 5.0))
for name in names:
    x = sensitivity_data[name]['x']
    dy = sensitivity_data[name]['dy']
    plt.plot(x, dy, label=name)

plt.xlabel('gamma_eff')
plt.ylabel('dS / dgamma_eff')
plt.title('Sensitivity along the effective noise direction')
plt.legend()
plt.tight_layout()
plt.show()


## Plot second derivatives

In [ ]:
plt.figure(figsize=(7.4, 5.0))
for name in names:
    x = sensitivity_data[name]['x']
    d2y = sensitivity_data[name]['d2y']
    plt.plot(x, d2y, label=name)

plt.xlabel('gamma_eff')
plt.ylabel('d²S / dgamma_eff²')
plt.title('Curvature along the effective noise direction')
plt.legend()
plt.tight_layout()
plt.show()


## Mark the rapid-degradation band

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

for name in names:
    x = sensitivity_data[name]['x']
    y = sensitivity_data[name]['y']
    x_drop = sensitivity_data[name]['peak_drop_x']

    axes[0].plot(x, y, label=name)
    axes[0].axvline(x_drop, linestyle='--', alpha=0.7)
    axes[1].plot(x, sensitivity_data[name]['dy'], label=name)
    axes[1].axvline(x_drop, linestyle='--', alpha=0.7)

axes[0].set_xlabel('gamma_eff')
axes[0].set_ylabel('Normalized order parameter')
axes[0].set_title('Response curves with fastest-drop markers')
axes[0].legend()

axes[1].set_xlabel('gamma_eff')
axes[1].set_ylabel('dS / dgamma_eff')
axes[1].set_title('Sensitivity curves with fastest-drop markers')
axes[1].legend()

plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
lines = []
lines.append("Effective noise direction and sensitivity summary")
lines.append("")
lines.append(f"Protocol: T={opt['T']:.4f}, alpha={opt['alpha']:.4f}, Omega={opt['omega_max']:.4f}, Delta0={opt['delta0']:.4f}, p={opt['p']:.4f}, q={opt['q']:.4f}")
lines.append("")

for name in names:
    lam = fit_results[name]['lambda']
    loss = fit_results[name]['loss']
    peak_drop_x = sensitivity_data[name]['peak_drop_x']
    peak_curv_x = sensitivity_data[name]['peak_curv_x']
    lines.append(
        f"- {name}: lambda={lam:.4f}, axis-loss={loss:.6e}, "
        f"fastest-drop gamma_eff={peak_drop_x:.4f}, strongest-curvature gamma_eff={peak_curv_x:.4f}"
    )

lines.append("")
lines.append("Interpretation:")
lines.append("- If lambda stays similar across metrics, the effective noise direction is structural.")
lines.append("- The first derivative identifies where phase-lock quality degrades fastest.")
lines.append("- The second derivative helps identify the onset of the rapid-degradation band.")
lines.append("- Together these define a directional sensitivity picture rather than only a static phase diagram.")

summary_text = "\n".join(lines)
print(summary_text)


## Final conclusion

This notebook upgrades the phase-lock story from a static map to a directional sensitivity analysis.

It shows:
- whether the fitted effective noise direction is stable across different order parameters,
- how the 1D response changes along that direction,
- and where the phase-locked CZ quality degrades fastest.

That gives the cleanest next-level interpretation:

**the noisy phase-locked CZ regime is organized by a dominant effective noise direction, and its loss is concentrated in a narrow rapid-degradation band along that direction.**
